In [58]:
xm_path = '../../data/XM-API/variable_query/2024-05-06_2024-04-29/'
import pandas as pd
cap = pd.read_csv(xm_path+'CapEfecNeta_Recurso.csv')
lis = pd.read_csv(xm_path+'ListadoRecursos_Sistema.csv')
lis = pd.merge(left=lis, right=cap, left_on='Values_Code', right_on='Code', how='inner')
lis = lis.groupby(['Values_Code','Values_Name','Values_Type', 'Values_Disp',
       'Values_RecType', 'Values_EnerSource',
       'Values_OperStartdate', 'Values_State']).agg({
       'Value' : 'mean' }).reset_index()

print('Menores de Agua', lis[(lis['Value'] < 20000) & (lis['Values_EnerSource'] == 'AGUA')].shape)
print('Mayores de Agua', lis[(lis['Value'] > 20000) & (lis['Values_EnerSource'] == 'AGUA')].shape)
print('AGUA', lis[lis['Values_EnerSource'] == 'AGUA'].shape)
lis.to_csv('delete.csv')

minors = lis[(lis['Value'] < 20000) & (lis['Values_EnerSource'] == 'AGUA')]
minors = minors[['Values_Code']]

map = pd.read_csv('../../data/XM-API/Map.csv')
map = pd.merge(left=map, right=minors, on='Values_Code', how='inner')
map = map[['GENERATION_PROJECT']]

geninfo = pd.read_csv('../../model/inputs/gen_infoa.csv')
# Primero creamos una lista de los proyectos de generación que están en 'map'
water_projects_to_update = map['GENERATION_PROJECT'].unique().tolist()

# Luego actualizamos gen_info
geninfo['gen_is_variable'] = geninfo.apply(
    lambda row: 1 if (row['gen_energy_source'] == 'Water' and 
                     row['GENERATION_PROJECT'] in water_projects_to_update) 
               else row['gen_is_variable'], 
    axis=1
)

geninfo.to_csv('../../model/inputs/gen_info.csv', index=False)
print(geninfo['gen_is_variable'].sum())

Menores de Agua (127, 9)
Mayores de Agua (30, 9)
AGUA (157, 9)
199


## Wind Dispatch

In [35]:
import pandas as pd
import swcol as sw

scenario_path = '../../model/'
scenario_path = '../../data/Colombia/scenarios/1/'
model_outputs_path = scenario_path+'outputs/'
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

# Load and preprocess data
sce_sw = pd.read_csv(model_outputs_path + '/dispatch.csv')
sce_sw['year'] = sce_sw['timestamp'].str[:4]
sce_sw = sce_sw[~sce_sw['timestamp'].str.contains('^2023')]
sce_sw['Energy_TWh_typical_yr'] = sce_sw['Energy_GWh_typical_yr'] / 1000
sce_sw.rename(columns={'gen_tech': 'Tech'}, inplace=True)

# Optionally map tech names if a template is provided
parse_tech, _, _, _ = sw.template.get()
sce_sw['Tech'] = sce_sw['Tech'].replace(parse_tech)

# Group and sum the energy by year and tech
# print(sce_sw.head(3))
#table = sce_sw.groupby(['timestamp', 'Tech'])['Energy_GWh_typical_yr'].sum().reset_index()
table = sce_sw.groupby(['timestamp', 'Tech']).agg({
    'Energy_GWh_typical_yr': 'sum'
}).reset_index()

print(table.columns)
# Optional: sort for cleaner display
#table = table.sort_values(by=['year', 'Energy_TWh_typical_yr'], ascending=[True, False])

# Display the table

#table = table[table['Tech'] == 'Hydro']

import plotly.express as px
fig = px.line(
    table, x='timestamp', y='Energy_GWh_typical_yr',
    color='Tech', title='Generation Dispatch Over Time by Technology',
    labels={'Energy_GWh_typical_yr': 'Dispatched Generation (GWh)'},
    color_discrete_map=tech_colors,
    category_orders={"Tech": tech_order},
    height=9*50, width=16*50,
    template='plotly_white',)

fig.show()

table.head(10)

Index(['timestamp', 'Tech', 'Energy_GWh_typical_yr'], dtype='object')


,timestamp,Tech,Energy_GWh_typical_yr
0,2024_Q1_holidays_0h,Biomass,2.952834e-10
1,2024_Q1_holidays_0h,Geothermal,2.951746e-10
2,2024_Q1_holidays_0h,Hydro,8.123360e+01
3,2024_Q1_holidays_0h,Run of River,4.467680e+00
4,2024_Q1_holidays_0h,Solar,1.852671e+00
5,2024_Q1_holidays_0h,Thermal,1.269044e+01
6,2024_Q1_holidays_0h,Wind,1.268649e+01
7,2024_Q1_holidays_10h,Biomass,2.952820e-10
8,2024_Q1_holidays_10h,Geothermal,2.951733e-10
9,2024_Q1_holidays_10h,Hydro,7.512835e+01


In [15]:
table.head(10)

,year,Tech,generation_project,Energy_TWh_typical_yr
266,2024,Wind,E_Caribe,7.503594
267,2024,Wind,E_GuajiraI,0.134388
268,2024,Wind,E_Wesp01,0.079891
535,2025,Wind,E_Caribe,7.503594
536,2025,Wind,E_GuajiraI,0.134388
537,2025,Wind,E_Wesp01,0.079891
806,2026,Wind,E_Caribe,7.503594
807,2026,Wind,E_GuajiraI,0.134388
808,2026,Wind,E_Wesp01,0.079891
1078,2027,Wind,E_Caribe,7.503594


## Only certain years

In [61]:
years = '2023|2025|2030|2035|2040|2045|2050'

model_inputs_path = '../../model/inputs/'
timepoints = pd.read_csv(model_inputs_path + '/timepoints.csv')

# Filtrar por años 2023 y 2024
filtered = timepoints[timepoints['timeseries'].str.contains(years)]
print("timepoints", timepoints.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'timepoints.csv', index=False)

filters = filtered[['timepoint_id']]

loads = pd.read_csv(model_inputs_path + '/loads.csv')
filtered = pd.merge(left=loads, right=filters, left_on='timepoints', right_on='timepoint_id')
filtered.drop(columns=['timepoint_id'], inplace=True)
print("loads", loads.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'loads.csv', index=False)

variable_capacity_factors = pd.read_csv(model_inputs_path + '/variable_capacity_factors.csv')
filtered = pd.merge(left=variable_capacity_factors, right=filters, on='timepoint_id')
print("variable_capacity_factors", variable_capacity_factors.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'variable_capacity_factors.csv', index=False)

timeseries = pd.read_csv(model_inputs_path + '/timeseries.csv')
filtered = timeseries[timeseries['TIMESERIES'].str.contains(years)]
print("timeseries", timeseries.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'timeseries.csv', index=False)

hydro_timeseries = pd.read_csv(model_inputs_path + '/hydro_timeseries.csv')
filtered = hydro_timeseries[hydro_timeseries['timeseries'].str.contains(years)]
print("hydro_timeseries", hydro_timeseries.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'hydro_timeseries.csv', index=False)

fuel_cost = pd.read_csv(model_inputs_path + '/fuel_cost.csv')
fuel_cost['period'] = fuel_cost['period'].astype(str)
filtered = fuel_cost[fuel_cost['period'].str.contains(years)]
print("fuel_cost", fuel_cost.shape, " - ",filtered.shape)
filtered.to_csv(model_inputs_path + 'fuel_cost.csv', index=False)

filters

timepoints (1344, 3)  -  (1344, 3)
loads (26880, 3)  -  (6720, 3)
variable_capacity_factors (1057536, 3)  -  (260736, 3)
timeseries (224, 5)  -  (56, 5)
hydro_timeseries (6720, 4)  -  (1680, 4)
fuel_cost (392, 4)  -  (98, 4)


,timepoint_id
0,1
1,2
2,3
3,4
4,5
...,...
1339,5372
1340,5373
1341,5374
1342,5375


fuel_cost (42, 4)  -  (28, 4)
